# 64장 · LangGraph interrupt와 승인 후 재개

승인이 필요한 그래프는 외부 작업 직전에 실행을 멈추고 상태를 저장합니다. 승인 또는 거절을 전달하면 그 실행을 이어갑니다.

과제 안내를 읽은 학생이 ‘등록 오류로 문의를 남겨 주세요’라고 했습니다. 내용을 보여 주는 순간에 문의를 등록해 버리면 학생은 취소할 기회를 잃습니다. LangGraph에서 실제로 멈춘 다음 같은 실행을 재개해 봅니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16, langgraph-checkpoint 4.3.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## interrupt()로 승인 대기

아래 그래프의 노드는 interrupt에서 실행을 멈춥니다. 체크포인터가 대기 중인 상태를 보관합니다. 여기서는 수업용 메모리를 쓰므로 커널을 다시 시작하면 기록이 사라집니다.

**승인 대기와 실행 재개**　게이트 앞에서 멈춘 요청은 승인 뒤 기록부로 이동합니다. interrupt() 직후와 재개 후에 등록 건수가 어떻게 달라지는지 비교하세요.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter64/index.html#concept-clip)


### 셀 01 · 승인 노드와 체크포인터 구성

노드에서 부수 효과가 생기는 줄은 interrupt 다음에만 둡니다. 같은 승인 번호로 두 번 저장하지 않도록 메모리 사전에서 확인합니다.

처음 interrupt에 닿으면 제안만 보이고 문의 등록은 0건입니다. 같은 작업을 재개하면 이 줄이 승인 또는 거절 값을 반환합니다. 승인 경로만 등록 줄을 지나므로 거절 뒤에는 등록 건수가 0으로 남습니다.


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class Request(TypedDict, total=False):
    approval_id: str
    text: str
    status: str

inquiries = {}
def approve(state: Request):
    decision = interrupt({"approval_id": state["approval_id"], "inquiry_text": state["text"]})
    if decision != "approve":
        return {"status": "declined"}
    inquiries.setdefault(state["approval_id"], state["text"])
    return {"status": "completed"}

builder = StateGraph(Request)
builder.add_node("confirm", approve)
builder.add_edge(START, "confirm")
builder.add_edge("confirm", END)
graph = builder.compile(checkpointer=InMemorySaver())

실행 순서를 연결했습니다. 아직 invoke하지 않았으므로 대기 중인 요청도, 등록된 문의도 없습니다.


### 셀 02 · interrupt()의 대기 상태

대기 응답이 보여 주는 본문과 실제로 기록된 문의 건수를 확인하세요.


In [2]:
config = {"configurable": {"thread_id": "chat-a-demo"}}
pending = graph.invoke({"approval_id": "A1", "text": "과제 등록 오류를 확인해 주세요."}, config)
print(pending["__interrupt__"][0].value["inquiry_text"])
print(len(inquiries))

과제 등록 오류를 확인해 주세요.
0


문의 본문 다음에 0이 나옵니다. 승인 화면에 문구를 띄울 수 있지만 등록 함수는 아직 호출되지 않았습니다.


## Command(resume=…)와 체크포인트

재개하려면 같은 thread_id로 체크포인트를 찾아야 합니다. 서비스에서는 여기에 소유자, 만료 시각, 승인 당시 인자의 해시도 대조합니다. 여기서는 로컬 그래프의 멈춤과 재개에 집중합니다.

<!-- lesson-visual practice-space -->
**승인 여부를 바꿔 같은 문의의 등록 건수 비교**

각 선택은 비어 있는 문의 저장소로 시작하는 독립 실행입니다. 같은 체크포인트의 A1에 전달할 결정을 바꿔 보세요.

대기와 거절은 등록 0건입니다. 승인으로 재개한 실행에서만 completed가 되고 문의가 1건 생깁니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter64/#practice-space)
<!-- /lesson-visual practice-space -->


### 셀 03 · Command(resume=…)로 승인 전달

기존 대기 요청에 Command를 보냅니다. 같은 질문을 새로 시작하지 않습니다.


In [3]:
done = graph.invoke(Command(resume="approve"), config)
print(done["status"], len(inquiries))

completed 1


completed 1입니다. 같은 승인 번호로 문의 한 건이 생겼습니다.


### 과제 04 · 거절 분기의 종료 상태

별도의 대기 요청 A2를 만들고 decline을 보내세요. 기록 목록에 A2가 남는지 확인하세요.


In [4]:
# 새 thread_id로 대기를 만들고 거절한 뒤 건수를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>other = {&#x27;configurable&#x27;: {&#x27;thread_id&#x27;: &#x27;chat-decline&#x27;}}
graph.invoke({&#x27;approval_id&#x27;: &#x27;A2&#x27;, &#x27;text&#x27;: &#x27;추가 문의&#x27;}, other)
refused = graph.invoke(Command(resume=&#x27;decline&#x27;), other)
print(refused[&#x27;status&#x27;], &#x27;A2&#x27; in inquiries)</code></pre>
<p>실행 결과</p><pre><code>declined False</code></pre>
<p>declined False가 보입니다. 같은 자료로 외부 문의가 실제 전송됐다고 말할 수는 없습니다.</p>
</details>


## 돌아보기

대기 전에 외부 등록 코드를 놓으면 재개 시 노드가 다시 실행될 때 문제가 생깁니다. 노드에서 어떤 줄이 다시 평가될 수 있는지 짚어 보세요.

### 대기와 재개의 시간 순서

**읽기와 제안**　동일 Q03을 읽고 원래 사용자·인자·만료를 묶습니다.

```
등록 0
```

**interrupt**　제안 객체를 보여 주고 checkpoint에 대기 상태를 남깁니다.

```
pending_approval / result=null
```

**결정 전달**　같은 요청·승인 ID·해시와 신뢰된 context로 Command를 보냅니다.

```
approve 또는 decline
```

**노드 재실행**　노드 앞부분부터 다시 실행하며 소유자·대상·만료를 확인합니다.

```
검사 후 write 또는 종료
```

**효과와 결과**　승인된 한 건만 등록하고 receipt를 A1과 분리합니다.

```
승인 A1 → 등록 1건
거절 A2 → 추가 등록 0건
```
